In [2]:
# prepare_rERP.ipynb
# Builds the long-format CSV that feeds the Julia rERP analysis: reads each
# participant's epochs, attaches Item/Condition from metadata and the norming
# predictor scores, and exports eeg_continuous_for_julia.csv.
#
# Inputs (EEG_DIR, STIMULI_FILE) are resolved via config.py, so they work from
# anywhere. Outputs are written with relative paths, so run this from the rERP
# folder if you want them to land next to rERP.jl and the R plotting scripts.

import mne
mne.set_log_level('error')
import numpy as np
import pandas as pd
import re
import sys
from pathlib import Path

# So it can find the other files
def find_project_root(marker="config.py"):
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} above {Path.cwd()}")

sys.path.insert(0, str(find_project_root()))
from config import EEG_DIR, BAD_SUBJ, STIMULI_FILE

In [3]:
# Cell 2: Config
# Inputs: EEG_DIR (epochs) and STIMULI_FILE (exp1 norming output) come from config.py
# Outputs stay local to this rERP folder (inside ./rERP_outputs/)

output_path = "./rERP_outputs/eeg_continuous_for_julia.csv"
Path("rERP_outputs").mkdir(exist_ok=True)

# Defined in config
bad_subjects = [int(re.search(r'\d+', s).group()) for s in BAD_SUBJ] 

rows_list_continuous = []
retention_report = []

In [4]:
# Read the exp1 output to check column names

df_stim = pd.read_csv(STIMULI_FILE)
print("Stimulus columns:", df_stim.columns.tolist())
print(df_stim.head())

Stimulus columns: ['id', 'Status_Final', 'Rank_Valid', 'Prop_Sens_A_OK', 'Prop_Sens_B_OK', 'Prop_Sens_AB_OK', 'Prop_Sens_C_Reject', 'Prop_Info_B_OK', 'Prop_Info_A_Reject', 'Mean_Sens_A', 'Mean_Sens_B', 'Mean_Sens_AB', 'Mean_Sens_C', 'Mean_Info_A', 'Mean_Info_B', 'Mean_Info_C', 'Information_Contrast_B_A', 'Semantic_Contrast_AB_C', 'Total_Prop_Visual_Error', 'Prop_Visual_Error_A', 'Prop_Visual_Error_B', 'Prop_Visual_Error_C', 'Mean_Visual_A', 'Mean_Visual_B', 'Mean_Visual_C']
    id     Status_Final  Rank_Valid  Prop_Sens_A_OK  Prop_Sens_B_OK  \
0    7  Selected_Tier_1           1        0.913043        0.960000   
1  140  Selected_Tier_1           2        0.956522        1.000000   
2  121  Selected_Tier_1           3        0.960000        0.965517   
3   18  Selected_Tier_1           4        1.000000        0.862069   
4   96  Selected_Tier_1           5        0.960000        0.896552   

   Prop_Sens_AB_OK  Prop_Sens_C_Reject  Prop_Info_B_OK  Prop_Info_A_Reject  \
0         0.9375

In [ ]:
# Extraction loop: it takes participants epoched files and extracts their Item/Condition labels through their metadata

# Searches for the -epo files in EEG_DIR, defined in config.py
epo_files = [
    f for f in EEG_DIR.rglob("*-epo.fif")
    if "alt_pipeline" not in f.parts
    and "baseline_check" not in f.parts
    and not f.name.endswith("-bc-epo.fif")
]

for epo_file in epo_files:
    participant_id = epo_file.parent.name
    participant_number = int(re.search(r'\d+', participant_id).group())

    # Skips participants marked as bad
    if participant_number in bad_subjects:
        print(f"Skipping {participant_id} (Bad subject)")
        continue

    print(f"Processing {participant_id}")

    # Load epochs into RAM
    epochs = mne.read_epochs(epo_file, preload=True, verbose=False)

    # Item/Condition are already attached as metadata in segmenting, it only retrieves it
    # (if it can't find any, it raises an error message)
    if epochs.metadata is None:
        print(f"❌ {participant_id}: no metadata — re-run segmenting. Skipping.")
        continue
    df_survivors = epochs.metadata.reset_index(drop=True)

    # Retention: how many trials (epochs) survived ICA + AutoReject for this participant.
    total_survivors = len(epochs)
    print(f"✅ {participant_id}: {total_survivors} epochs with metadata")
    retention_report.append({
        'Participant': participant_number,
        'Preserved': total_survivors,
    })

    # Build long-format EEG table
    electrodes = epochs.copy().pick_types(eeg=True).ch_names

    # Guard: must be exactly one survivor row per epoch, in order
    assert len(df_survivors) == len(epochs), (
        f"{participant_id}: {len(df_survivors)} survivor rows vs {len(epochs)} epochs"
    )

    df_epoch = epochs.copy().pick(electrodes).to_data_frame(time_format=None)
    df_epoch['Timestamp'] = np.round(df_epoch['time'] * 1000).astype(int)

    # Map metadata by positional epoch index.
    item_map = dict(enumerate(df_survivors['Item']))
    cond_map = dict(enumerate(df_survivors['Condition'].astype(str).str.strip().str.lower()))
    df_epoch['Item']      = df_epoch['epoch'].map(item_map)
    df_epoch['Condition'] = df_epoch['epoch'].map(cond_map)
    df_epoch['Subject']   = participant_number

    # Drop MNE's helper columns
    df_epoch = df_epoch.drop(columns=[c for c in ['epoch', 'time', 'condition']
                                      if c in df_epoch.columns])

    rows_list_continuous.append(df_epoch)

Processing participant_1
✅ participant_1: 200 epochs with metadata
Processing participant_10
✅ participant_10: 209 epochs with metadata
Processing participant_11
✅ participant_11: 199 epochs with metadata
Processing participant_12
✅ participant_12: 217 epochs with metadata
Processing participant_13
✅ participant_13: 135 epochs with metadata
Processing participant_14
✅ participant_14: 232 epochs with metadata
Processing participant_15
✅ participant_15: 230 epochs with metadata
Processing participant_16
✅ participant_16: 202 epochs with metadata
Processing participant_17
✅ participant_17: 216 epochs with metadata
Processing participant_18
✅ participant_18: 225 epochs with metadata
Processing participant_19
✅ participant_19: 240 epochs with metadata
Processing participant_2
✅ participant_2: 240 epochs with metadata
Processing participant_20
✅ participant_20: 211 epochs with metadata
Processing participant_21
✅ participant_21: 178 epochs with metadata
Processing participant_22
✅ participan

In [6]:
# Cell 5: Concatenates everything into a big table and merges it with stimulus scores, assigning per-condition averages

print("\nConcatenating...")
df_eeg_all = pd.concat(rows_list_continuous, ignore_index=True)

df_eeg_all['Item'] = df_eeg_all['Item'].astype(str)
df_stim['id'] = df_stim['id'].astype(str)

df_merged = pd.merge(df_eeg_all, df_stim, left_on='Item', right_on='id')

def assign_by_cond(row, a, b, c):
    cond = str(row['Condition']).strip().lower()
    return {'a': row[a], 'b': row[b], 'c': row[c]}.get(cond, np.nan)

print("Calculating dynamic scores")
df_merged['Semantic_Score']    = df_merged.apply(lambda r: assign_by_cond(r, 'Mean_Sens_A',   'Mean_Sens_B',   'Mean_Sens_C'),   axis=1)
df_merged['Info_Score']        = df_merged.apply(lambda r: assign_by_cond(r, 'Mean_Info_A',   'Mean_Info_B',   'Mean_Info_C'),   axis=1)
df_merged['Mean_Visual_Error'] = df_merged.apply(lambda r: assign_by_cond(r, 'Mean_Visual_A', 'Mean_Visual_B', 'Mean_Visual_C'), axis=1)

n_before = len(df_merged)
df_merged = df_merged.dropna(subset=['Semantic_Score', 'Info_Score', 'Mean_Visual_Error'])
if (n_dropped := n_before - len(df_merged)):
    print(f"⚠️ Dropped {n_dropped} rows with missing scores before export.")


Concatenating...
Calculating dynamic scores


In [7]:
# Cell 6: collinearity check + export (for the Dissertation)

df_unique_trials = df_merged[['Item', 'Condition', 'Semantic_Score', 'Info_Score', 'Mean_Visual_Error']].drop_duplicates()
matrice_corr = df_unique_trials[['Semantic_Score', 'Info_Score', 'Mean_Visual_Error']].corr(method='pearson')
print("\nCorrelation Matrix"); print(matrice_corr)
df_unique_trials.to_csv("./rERP_outputs/predictor_values.csv", index=False)

columns_to_keep = ['Subject', 'Item', 'Condition', 'Timestamp',
                   'Semantic_Score', 'Info_Score', 'Mean_Visual_Error'] + electrodes
df_merged[columns_to_keep].to_csv(output_path, index=False)
print(f"File ready for Julia analysis: {output_path}")


Correlation Matrix
                   Semantic_Score  Info_Score  Mean_Visual_Error
Semantic_Score           1.000000    0.708469           0.665157
Info_Score               0.708469    1.000000           0.494734
Mean_Visual_Error        0.665157    0.494734           1.000000
File ready for Julia analysis: ./rERP_outputs/eeg_continuous_for_julia.csv


In [8]:
# Cell 7: VIF diagnostics for the Dissertation

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X = add_constant(df_unique_trials[['Semantic_Score', 'Info_Score', 'Mean_Visual_Error']])
vif = pd.Series([variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
                index=X.columns).drop('const')

diagnostics = matrice_corr.round(2).copy()
diagnostics['VIF'] = vif.round(2)
diagnostics.to_csv("./rERP_outputs/predictor_diagnostics.csv")
print(diagnostics)

                   Semantic_Score  Info_Score  Mean_Visual_Error   VIF
Semantic_Score               1.00        0.71               0.67  2.72
Info_Score                   0.71        1.00               0.49  2.01
Mean_Visual_Error            0.67        0.49               1.00  1.80


Visual error is independent from the others, but semantic_score is linked to info_score (what is a bit obvious, because what is unintelligible can't add much information). There's no BIG problem because the rERP tries to isolate the unique contributions, but it is something worth noting.